In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split

In [2]:
books_url = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master/books.csv"

books = pd.read_csv(books_url)
books = books[["book_id", "title", "authors", "average_rating", "ratings_count"]]
books = books.dropna().reset_index(drop=True)

print("Розмір даних:", books.shape)
display(books.head())

Розмір даних: (10000, 5)


,book_id,title,authors,average_rating,ratings_count
0,1,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,4.34,4780653
1,2,Harry Potter and the Sorcerer's Stone (Harry P...,"J.K. Rowling, Mary GrandPré",4.44,4602479
2,3,"Twilight (Twilight, #1)",Stephenie Meyer,3.57,3866839
3,4,To Kill a Mockingbird,Harper Lee,4.25,3198671
4,5,The Great Gatsby,F. Scott Fitzgerald,3.89,2683664


In [3]:
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(books["authors"])
print("Розмір TF-IDF матриці:", tfidf_matrix.shape)

# індекси книг за назвою
title_to_idx = pd.Series(books.index, index=books["title"]).drop_duplicates()


def recommend(title, n=5):
    if title not in title_to_idx:
        return f"Книгу '{title}' не знайдено"
    idx = title_to_idx[title]
    # косинусна подібність однієї книги до всіх інших
    sim = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    top_idx = sim.argsort()[::-1]
    top_idx = [i for i in top_idx if i != idx][:n]
    result = books.iloc[top_idx][["title", "authors"]].copy()
    result["similarity"] = sim[top_idx].round(3)
    return result

Розмір TF-IDF матриці: (10000, 6193)


In [4]:
for t in ["The Hunger Games", "Harry Potter and the Philosopher's Stone", "The Hobbit"]:
    print(f"\nСхожі на: {t}")
    display(recommend(t, n=5))


Схожі на: The Hunger Games


"Книгу 'The Hunger Games' не знайдено"


Схожі на: Harry Potter and the Philosopher's Stone


"Книгу 'Harry Potter and the Philosopher's Stone' не знайдено"


Схожі на: The Hobbit


,title,authors,similarity
963,J.R.R. Tolkien 4-Book Boxed Set: The Hobbit an...,J.R.R. Tolkien,1.0
188,"The Lord of the Rings (The Lord of the Rings, ...",J.R.R. Tolkien,1.0
160,"The Return of the King (The Lord of the Rings,...",J.R.R. Tolkien,1.0
154,"The Two Towers (The Lord of the Rings, #2)",J.R.R. Tolkien,1.0
18,The Fellowship of the Ring (The Lord of the Ri...,J.R.R. Tolkien,1.0


In [5]:
ratings_url = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master/ratings.csv"

ratings = pd.read_csv(ratings_url)
print("Розмір ratings:", ratings.shape)
display(ratings.head())

reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings[["user_id", "book_id", "rating"]], reader)

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

Розмір ratings: (5976479, 3)


,user_id,book_id,rating
0,1,258,5
1,2,4081,4
2,2,260,5
3,2,9296,5
4,2,2318,3


In [6]:
svd = SVD(random_state=42)
svd.fit(trainset)

predictions = svd.test(testset)
rmse = accuracy.rmse(predictions)

RMSE: 0.8310


In [8]:
user_id = 1  # можна змінити на будь-якого користувача з ratings

# книги, які користувач ще не оцінив
rated = set(ratings.loc[ratings["user_id"] == user_id, "book_id"])
candidates = [b for b in books["book_id"] if b not in rated]

# передбачувані рейтинги
preds = [(b, svd.predict(user_id, b).est) for b in candidates]
preds_df = pd.DataFrame(preds, columns=["book_id", "predicted_rating"])

recommendations = (
    preds_df.sort_values("predicted_rating", ascending=False)
    .merge(books[["book_id", "title", "authors"]], on="book_id")
    .reset_index(drop=True)
)
recommendations["predicted_rating"] = recommendations["predicted_rating"].round(3)

# 5 найкращих рекомендацій
top5 = recommendations.head(5)
print(f"Топ-5 рекомендацій для користувача {user_id}:")
display(top5)

# збереження у файл
top5.to_csv("book_recommendations.csv", index=False, encoding="utf-8-sig")
print("Файл book_recommendations.csv збережено")

Топ-5 рекомендацій для користувача 1:


,book_id,predicted_rating,title,authors
0,8946,4.749,The Divan,Hafez
1,3628,4.669,The Complete Calvin and Hobbes,Bill Watterson
2,6590,4.652,The Authoritative Calvin and Hobbes: A Calvin ...,Bill Watterson
3,852,4.612,When Breath Becomes Air,"Paul Kalanithi, Abraham Verghese"
4,7254,4.589,Homicidal Psycho Jungle Cat: A Calvin and Hobb...,Bill Watterson


Файл book_recommendations.csv збережено
